## Extract Junior Authors from Matched Awards
The 246 matched awards to identify junior authors

In [ ]:
import pandas as pd
import requests
import json
import time

# Load your helper functions from demo notebook
def get_author_works(author_id):
    """Retrieve all works for a given author with pagination."""
    all_works = []
    cursor = "*"
    
    while cursor:
        url = "https://api.openalex.org/works"
        params = {
            "filter": f"author.id:{author_id}",
            "per-page": 200,
            "cursor": cursor,
            "mailto": "shaheryar.4822@student.uu.se"
        }
        
        response = requests.get(url, params=params)
        data = response.json()
        results = data.get("results", [])
        all_works.extend(results)
        
        cursor = data.get("meta", {}).get("next_cursor")
        if cursor:
            time.sleep(0.05)
    
    return all_works

def calculate_career_age(works, reference_year):
    """Calculate years since first publication."""
    years = [w['publication_year'] for w in works if w.get('publication_year')]
    if not years:
        return None
    first_year = min(years)
    return reference_year - first_year

# Load matched awards
matched_df = pd.read_csv("matched_awards_2010_2018.csv")

# Parse authorships back from JSON
matched_df['authorships'] = matched_df['authorships'].apply(json.loads)

print(f"{'='*60}")
print(f"EXTRACTING JUNIOR AUTHORS")
print(f"{'='*60}\n")
print(f"Processing {len(matched_df)} matched awards...")
print(f"This will take ~2-4 hours (API calls for each author)\n")

junior_authors = []
processed_authors = set()  # Track to avoid duplicates

for idx, row in matched_df.iterrows():
    work_id = row['openalex_id']
    award_year = int(row['year'])
    conference = row['conference']
    
    print(f"[{idx+1}/{len(matched_df)}] {conference} {award_year}: {row['openalex_title'][:50]}...")
    
    # Get top 3 authorships only (to identify junior authors)
    authorships = row['authorships'][:3]
    
    for position, authorship in enumerate(authorships, 1):
        author = authorship['author']
        author_id = author['id']
        author_name = author['display_name']
        
        # Skip if already processed this author
        if author_id in processed_authors:
            print(f"  ⏭️  {author_name} (already processed)")
            continue
        
        processed_authors.add(author_id)
        
        # Fetch author's publication history
        print(f"  🔍 {author_name} (pos {position})...")
        author_works = get_author_works(author_id)
        career_age = calculate_career_age(author_works, award_year)
        
        # Check if junior (≤5 years)
        if career_age is not None and career_age <= 5:
            # Extract institutions
            institutions = [
                {
                    'id': inst.get('id', ''),
                    'name': inst.get('display_name', ''),
                    'country': inst.get('country_code', '')
                }
                for inst in authorship.get('institutions', [])
            ]
            
            junior_authors.append({
                'award_id': row['award_id'],
                'conference': conference,
                'award_year': award_year,
                'work_id': work_id,
                'award_title': row['award_title'],
                'author_id': author_id,
                'author_name': author_name,
                'author_position': position,
                'career_age_at_award': career_age,
                'total_pubs_at_award': len(author_works),
                'institutions': json.dumps(institutions),
                'is_corresponding': authorship.get('is_corresponding', False)
            })
            
            print(f"    ✅ JUNIOR: {author_name} (age {career_age}y, {len(author_works)} pubs)")
        elif career_age is not None:
            print(f"    ❌ Senior: {author_name} (age {career_age}y)")
        else:
            print(f"    ⚠️  No career data: {author_name}")
        
        time.sleep(0.1)  # Rate limiting
    
    # Save progress every 25 awards
    if (idx + 1) % 25 == 0:
        pd.DataFrame(junior_authors).to_csv("junior_authors_progress.csv", index=False)
        print(f"\n💾 Progress: {len(junior_authors)} junior authors found from {idx+1} awards\n")

# Final save
junior_df = pd.DataFrame(junior_authors)
junior_df.to_csv("junior_authors_2010_2018.csv", index=False)

print(f"\n{'='*60}")
print(f"EXTRACTION COMPLETE")
print(f"{'='*60}")
print(f"Junior authors found: {len(junior_df)}")
print(f"From {len(matched_df)} awards")
print(f"Average per award: {len(junior_df)/len(matched_df):.2f}")
print(f"\nCareer age distribution:")
print(junior_df['career_age_at_award'].value_counts().sort_index())
print(f"\nPosition distribution:")
print(junior_df['author_position'].value_counts())
print(f"\nTop institutions:")
inst_counts = {}
for insts_json in junior_df['institutions']:
    insts = json.loads(insts_json)
    for inst in insts:
        name = inst.get('name', 'Unknown')
        inst_counts[name] = inst_counts.get(name, 0) + 1

top_insts = sorted(inst_counts.items(), key=lambda x: x[1], reverse=True)[:10]
for name, count in top_insts:
    print(f"  {name}: {count}")


EXTRACTING JUNIOR AUTHORS

Processing 246 matched awards...
This will take ~2-4 hours (API calls for each author)

[1/246] CHI 2018: Agile 3D Sketching with Air Scaffolding...
  🔍 Yongkwan Kim (pos 1)...
    ❌ Senior: Yongkwan Kim (age 13y)
  🔍 Sang-Gyun An (pos 2)...
    ✅ JUNIOR: Sang-Gyun An (age 1y, 8 pubs)
  🔍 Joon Hyub Lee (pos 3)...
    ❌ Senior: Joon Hyub Lee (age 6y)
[2/246] CHI 2018: Addressing Age-Related Bias in Sentiment Analysis...
  🔍 Mark Díaz (pos 1)...
    ❌ Senior: Mark Díaz (age 15y)
  🔍 Isaac Johnson (pos 2)...
    ❌ Senior: Isaac Johnson (age 14y)
  🔍 Amanda Lazar (pos 3)...
    ❌ Senior: Amanda Lazar (age 45y)
[3/246] CHI 2018: How Relevant are Incidental Power Poses for HCI?...
  🔍 Yvonne Jansen (pos 1)...
    ❌ Senior: Yvonne Jansen (age 11y)
  🔍 Kasper Hornbæk (pos 2)...
    ❌ Senior: Kasper Hornbæk (age 19y)
[4/246] CHI 2018: Voice Interfaces in Everyday Life...
  🔍 Martin Porcheron (pos 1)...
    ❌ Senior: Martin Porcheron (age 22y)
  🔍 Joel E. Fischer (pos 